### Step 2: Base model with real data

Same model as Step 1: same decision, same 3 constraints, same objective. The only change is that the numbers now come from Ed's spreadsheet, `NFL Ad PKG MAIN.xlsx`, instead of being typed in.

**The scenario:** a CBS affiliate sells a season-long package: one :30 ad in each of 27 NFL games, for \$108,000. It's July, before the NFL schedule is released. The station must decide which prospects to sell to, and how much audience to guarantee each one.

**How Step 1 maps onto the real data:**

| Step 1 (base model) | Step 2 (real data) | Where it comes from |
| --- | --- | --- |
| 30 customers, goal CPM | Same 30 prospects and goal CPMs | Prospect Data Table |
| Penalty: \$0.05-\$0.25 per missing viewer | Makegood multiplier: 2x, 3x, or 4x the value of the missing audience | Prospect Data Table |
| 4 packages, each with its own views and price | 4 guarantee levels on the same 27-game, \$108,000 package | Pre-Schedule Release (team picks the levels) |
| 5 actual-viewer outcomes (cumulative probabilities converted) | 3 Nielsen increases: +15% / +20% / +25%, with probabilities 50% / 35% / 15% | Schedule Release 1-Min Qualifier |
| 20 spots | 20 packages | Ed |

**The packages are guarantee levels.** The station always sells the same 27 games for \$108,000. What changes is how much audience it promises:

| Package | Guarantee above baseline | Audience promised (000s) | CPM to the customer |
| --- | ---: | ---: | ---: |
| conservative | +0% | 1,631.3 | \$66.20 |
| medium | +15% | 1,876.0 | \$57.57 |
| premium | +20% | 1,957.6 | \$55.17 |
| aggressive | +25% | 2,039.1 | \$52.96 |

Promising more audience for the same price lowers the CPM, so more prospects are willing to buy. But the actual audience only grows by 15%, 20%, or 25%, so a bigger promise is more likely to fall short.

**Units:** audiences stay in thousands of households ("000s"), matching the spreadsheet. CPM is dollars per thousand, so 000s × CPM = dollars.

**Penalty:** if the station falls short, the customer is owed the value of the missing audience (shortfall × package CPM), times their multiplier.

**Assumptions (to confirm with Ed):**
1. Guarantee levels of 0%, 15%, 20%, and 25%. We chose these to match the three Nielsen increases, plus no increase. The Plan B notebook used 0 / 15 / 25%.
2. The missing audience is valued at the package CPM.
3. This step uses the July view: generic games, before the dud-game cuts. The real schedule comes in a later step.

In [ ]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import course_utils  # course helpers (print_status, print_model, ...)

In [ ]:
# ============================================================
# LOAD THE DATA
# ============================================================

# Ed's spreadsheet. Keep it in the same folder as this notebook.
# We only read from it; we never edit it.

DATA_FILE = "NFL Ad PKG MAIN.xlsx"


# ------------------------------------------------------------
# A. The package: 'Pre-Schedule Release', rows 13-17
# ------------------------------------------------------------

# One row per game type (Thanksgiving, noon regional, ...).
# This is what the station knows in July, before the schedule is out.
#
# col C = game type
# col E = number of games
# col F = rate per spot ($)
# col H = audience per game (000s = thousands of households)

df_games = pd.read_excel(
    DATA_FILE,
    sheet_name="Pre-Schedule Release",
    header=None,
    skiprows=12,
    nrows=5,
    usecols="C,E,F,H"
)

df_games.columns = ["game_type", "num_games", "rate", "audience_000s"]

package_price = (df_games["num_games"] * df_games["rate"]).sum()
baseline_000s = (df_games["num_games"] * df_games["audience_000s"]).sum()

# Check against the spreadsheet's own totals (row 18)
assert df_games["num_games"].sum() == 27
assert package_price == 108_000
assert abs(baseline_000s - 1631.3) < 0.001


# ------------------------------------------------------------
# B. Ratings scenarios: 'Schedule Release 1-Min Qualifer', T6:U8
# ------------------------------------------------------------

# Nielsen's new 1-minute rule raises ratings, but nobody knows
# by how much. Management gives 3 possible increases.
#
# col T = label, such as "15% Increase"
# col U = probability of that increase

df_scen = pd.read_excel(
    DATA_FILE,
    sheet_name="Schedule Release 1-Min Qualifer",
    header=None,
    skiprows=5,
    nrows=3,
    usecols="T:U"
)

df_scen.columns = ["label", "probability"]

# "15% Increase" -> 0.15
df_scen["increase"] = df_scen["label"].str.split("%").str[0].astype(float) / 100

# Check: probabilities add up to 100%
assert abs(df_scen["probability"].sum() - 1.0) < 0.000001


# ------------------------------------------------------------
# C. Prospects: 'Prospect Data Table', rows 4-33
# ------------------------------------------------------------

# col B = prospect ID
# col C = prospect name
# col D = salesperson        (not used until a later step)
# col E = close ratio        (not used until a later step)
# col F = goal CPM
# col G = makegood penalty multiplier (2, 3, or 4)

df_p = pd.read_excel(
    DATA_FILE,
    sheet_name="Prospect Data Table",
    skiprows=2,
    nrows=30,
    usecols="B:G"
)

df_p.columns = ["id", "prospect", "salesperson", "close_ratio",
                "goal_cpm", "penalty"]

# Check: 30 prospects, 6 salespeople with 5 prospects each
assert len(df_p) == 30
assert df_p["salesperson"].value_counts().tolist() == [5] * 6


print(f"Package price:  ${package_price:,.0f}")
print(f"Baseline 000s:  {baseline_000s:,.1f}")
print()
print(df_scen)
print()
df_p

In [ ]:
# ============================================================
# 1. INDEX SETS
# ============================================================

# customers: the 30 prospects, read from the spreadsheet

customers = df_p["id"].tolist()

# packages: how much audience we guarantee, above the baseline.
# Same 4 names as the base model. The levels match the
# 3 possible Nielsen increases, plus no increase at all.

packages = [
    "conservative",
    "medium",
    "premium",
    "aggressive"
]

# scenarios: the 3 possible Nielsen increases, from the spreadsheet

scenarios = df_scen["label"].tolist()


# ============================================================
# 2. CUSTOMER PARAMETERS
# ============================================================

goal_cpm = dict(zip(df_p["id"], df_p["goal_cpm"]))

# Makegood multiplier: if we fall short, customer c is owed
# penalty[c] times the value of the missing audience.

underdelivery_penalty = dict(zip(df_p["id"], df_p["penalty"]))


# ============================================================
# 3. AD PACKAGE PARAMETERS
# ============================================================

# Guarantee level for each package (TEAM ASSUMPTION)

guarantee = {
    "conservative": 0.00,
    "medium": 0.15,
    "premium": 0.20,
    "aggressive": 0.25
}

# Audience we promise the customer (000s)

guaranteed_000s = {}

for p in packages:
    guaranteed_000s[p] = baseline_000s * (1 + guarantee[p])

# Every package is the same 27 games at the same price.
# A bigger guarantee means a lower CPM for the customer.

price = {}
cpm = {}

for p in packages:
    price[p] = package_price
    cpm[p] = price[p] / guaranteed_000s[p]


# ============================================================
# 4. ACTUAL AUDIENCE OUTCOMES
# ============================================================

# Unlike the base model, these are 3 separate outcomes that
# already add up to 100%, so no conversion is needed.
#
# Actual audience = baseline x (1 + Nielsen increase)

probability = dict(zip(df_scen["label"], df_scen["probability"]))

actual_000s = {}

for s in scenarios:
    increase = df_scen.loc[df_scen["label"] == s, "increase"].iloc[0]
    actual_000s[s] = baseline_000s * (1 + increase)


# ============================================================
# 5. DETERMINE PACKAGE ELIGIBILITY
# ============================================================

# eligible[c,p] = 1 if customer c's goal CPM is high enough
# to accept package p.
#
# Otherwise eligible[c,p] = 0.

eligible = {}

for c in customers:
    for p in packages:

        if cpm[p] <= goal_cpm[c]:
            eligible[c, p] = 1
        else:
            eligible[c, p] = 0


# ============================================================
# 6. CALCULATE EXPECTED UNDERDELIVERY PENALTY
# ============================================================

# Shortfall (000s) x package CPM ($ per 000) = value of the
# missing audience in dollars. The customer is owed that
# value times their penalty multiplier.

expected_penalty = {}

for c in customers:
    for p in packages:

        expected_penalty[c, p] = sum(
            probability[s] *
            max(guaranteed_000s[p] - actual_000s[s], 0) *
            cpm[p] *
            underdelivery_penalty[c]
            for s in scenarios
        )


# ============================================================
# 7. CALCULATE EXPECTED PROFIT
# ============================================================

expected_profit = {}

for c in customers:
    for p in packages:

        expected_profit[c, p] = (
            price[p]
            - expected_penalty[c, p]
        )


# ============================================================
# 8. CREATE GUROBI MODEL
# ============================================================

model = gp.Model("NFL_Advertising_Optimization_Real_Data")


# ============================================================
# 9. DECISION VARIABLES
# ============================================================

# x[c,p] = 1 if customer c receives package p
#          0 otherwise

x = model.addVars(
    customers,
    packages,
    vtype=GRB.BINARY,
    name="x"
)


# ============================================================
# 10. CONSTRAINTS
# ============================================================

# ------------------------------------------------------------
# Constraint 1:
# Each customer can receive at most one package.
# ------------------------------------------------------------

model.addConstrs(
    (
        gp.quicksum(x[c, p] for p in packages) <= 1
        for c in customers
    ),
    name="one_package_per_customer"
)


# ------------------------------------------------------------
# Constraint 2:
# Package CPM cannot exceed customer's goal CPM.
# ------------------------------------------------------------

model.addConstrs(
    (
        x[c, p] <= eligible[c, p]
        for c in customers
        for p in packages
    ),
    name="cpm_requirement"
)


# ------------------------------------------------------------
# Constraint 3:
# We have 20 advertising spots, so we can sell at most 20 packages.
# ------------------------------------------------------------

model.addConstr(
    gp.quicksum(
        x[c, p]
        for c in customers
        for p in packages
    ) <= 20,
    name="advertising_spots"
)


# ============================================================
# 11. OBJECTIVE FUNCTION
# ============================================================

# Maximize expected profit:
#
# Revenue from the package
# minus
# Expected underdelivery penalty

model.setObjective(
    gp.quicksum(
        expected_profit[c, p] * x[c, p]
        for c in customers
        for p in packages
    ),
    GRB.MAXIMIZE
)


# ============================================================
# 12. SOLVE MODEL
# ============================================================

model.optimize()


# ============================================================
# 13. DISPLAY RESULTS
# ============================================================

print("\nPACKAGE OPTIONS")
print("=" * 75)

for p in packages:

    num_eligible = sum(eligible[c, p] for c in customers)

    print(
        f"{p.capitalize():12} | "
        f"Guarantee: +{guarantee[p]:>4.0%} | "
        f"000s: {guaranteed_000s[p]:>7,.1f} | "
        f"CPM: ${cpm[p]:>5.2f} | "
        f"Eligible customers: {num_eligible:>2}"
    )

if model.status == GRB.OPTIMAL:

    print("\nOPTIMAL ADVERTISING PLAN")
    print("=" * 75)

    total_revenue = 0
    total_expected_penalty = 0
    packages_sold = 0

    for c in customers:
        for p in packages:

            if x[c, p].X > 0.5:

                revenue = price[p]
                penalty = expected_penalty[c, p]
                profit = expected_profit[c, p]

                total_revenue += revenue
                total_expected_penalty += penalty
                packages_sold += 1

                print(
                    f"{c}: "
                    f"{p.capitalize():12} | "
                    f"Goal CPM: ${goal_cpm[c]:>5.2f} | "
                    f"CPM: ${cpm[p]:>5.2f} | "
                    f"Penalty: {underdelivery_penalty[c]}x | "
                    f"Revenue: ${revenue:>9,.0f} | "
                    f"Expected Penalty: ${penalty:>9,.2f} | "
                    f"Expected Profit: ${profit:>9,.2f}"
                )

    print("=" * 75)

    print(f"Packages Sold:       {packages_sold} of 20")
    print(f"Total Revenue:       ${total_revenue:,.2f}")
    print(f"Expected Penalties:  ${total_expected_penalty:,.2f}")
    print(f"Expected Profit:     ${model.ObjVal:,.2f}")

else:

    print("No optimal solution found.")

### What the results show

- **Only 8 of 20 packages sell.** At a 0% guarantee the CPM is \$66.20, above every prospect's goal (the highest is \$58.81), so nobody buys it. Even at a 25% guarantee (\$52.96), only 8 prospects qualify.
- **Each customer gets the smallest guarantee that meets their goal CPM.** The price is the same for every package, so a bigger guarantee only adds risk. The model promises extra audience only when it's needed to make the sale.
- **Medium carries no risk.** The smallest Nielsen increase is 15%, so a 15% guarantee is always met.
- **Premium and aggressive carry risk.** For example, C21 on aggressive: the expected shortfall is 50% × 163.13 + 35% × 81.57 = 110.11 (000s). That's worth 110.11 × \$52.96 = \$5,832, and C21's 3x multiplier makes it \$17,496.
- **C29 barely qualifies for medium.** Its goal CPM is \$57.57, and the medium CPM is \$57.5694. A one-cent change would drop it to premium.

**Takeaway:** with real data, the goal CPM rule leaves most prospects out. Discounts (Step 3a) lower the CPM without promising more audience, which should let more prospects buy.

### Changes from Step 1

1. **Data is read from the spreadsheet** (the new data cell), with checks against the spreadsheet's own totals. Nothing is typed in.
2. **Packages are guarantee levels** on one \$108,000 package, instead of four packages with different prices. `price[p]` is the same for every package, and `cpm[p]` is calculated.
3. **Outcomes are the 3 Nielsen scenarios.** They already add up to 100%, so the cumulative conversion (and the 1.5M stand-in) is gone. `scenarios` is back as a name, now for real scenarios.
4. **The penalty is a multiplier** on the value of the missing audience, instead of dollars per missing viewer.
5. The model itself (sections 8-12: variables, constraints, objective) is **unchanged**. The display now also lists the package options and each customer's goal CPM.